In [1]:
# Import libraries for modeling, text processing, and validation
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier # Optimized Gradient Boosting classifier


# Load the 'Newsgroups' dataset, a collection of English texts classified by topic

data = fetch_20newsgroups(subset='all', remove=())  # The full dataset is loaded without removing any part

# Create a DataFrame with the document text and its numeric label
df = pd.DataFrame({
    'text': data.data,
    'original_label': data.target
})

# Add a column with the name of the textual category corresponding to each label
category_names = data.target_names
df['topic'] = df['original_label'].apply(lambda x: category_names[x])

# Definition of a binary variable "sport"
# Positive classes: categories related to sports and motor vehicles

sport_categories = ['rec.sport.baseball', 'rec.sport.hockey', 'rec.autos', 'rec.motorcycles']
df['sport'] = df['topic'].apply(lambda x: 1 if x in sport_categories else 0)

# Split the data into features (X) and binary labels (y)

X = df['text'].values
y = df['sport'].values

# Definition of a custom stopword list
# These words will be ignored during vectorization

stopwords = [
    'as', 'an', 'the', 'in', 'on', 'at', 'to', 'of', 'and', 'or',
    'is', 'it', 'for', 'with', 'that', 'this', 'was', 'be',
    'are', 'were', 'been', 'from', 'by', 'about', 'into', 'out',
    'up', 'down', 'over', 'under', 'then', 'than', 'so', 'but', 'not'
]

In [2]:

i=0# Definition of the range of values for parameter k (number of SVD components)
# and initialization of the list used to store the results

k_values = [100]  # Number of latent components to retain with SVD (LSA)
results = []    # List that will contain the mean performance for each value of k

# Configuration of stratified cross-validation with 5 splits
# The class proportion is preserved between training and test in each fold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Main iteration for each component value (k) to be applied with SVD

for k in k_values:
    scores = []  # List to store the AUC-ROC values for each fold

    for train_idx, test_idx in cv.split(X, y):
        # Split the data into training and test sets

        X_train_raw, X_test_raw = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        # Vectorize the text using TF-IDF
        # It is fitted only on the training set to avoid information leakage

        tfidf = TfidfVectorizer(max_features=10000)
        X_train_tfidf = tfidf.fit_transform(X_train_raw)
        X_test_tfidf = tfidf.transform(X_test_raw)

        # Dimensionality reduction with Truncated SVD (LSA)
        # The TF-IDF representation is reduced to k latent dimensions

        svd = TruncatedSVD(n_components=k, random_state=42)
        X_train_svd = svd.fit_transform(X_train_tfidf)
        X_test_svd = svd.transform(X_test_tfidf)

        # Train the XGBoost model for binary classification
        # Parameters:
        # - max_depth: maximum tree depth
        # - n_estimators: number of trees in the model 
        # - use_label_encoder: disabled to avoid old warnings
        # - eval_metric: metric used during training

        model = XGBClassifier(
            max_depth=6,
            n_estimators=500,
            use_label_encoder=False,
            eval_metric='logloss',
            random_state=42
        )
        model.fit(X_train_svd, y_train)
        print("Iteration: "+str(i) )  # Message indicating that one iteration has finished

        # Probability prediction and AUC-ROC metric computation
        # This metric measures the model's ability to separate the classes

        probs = model.predict_proba(X_test_svd)[:, 1]
        auc = roc_auc_score(y_test, probs)
        scores.append(auc)
        print("AUC-ROC: Fold " + str(i) +  ":" + str(auc))  # Print the AUC score of the current fold
        i=i+1
    # Once the 5 folds have been completed, the mean of the scores is computed
    # and the result is stored for the current value of k

    results.append({
        'k_components': k,
        'mean_auc_roc': np.mean(scores)
    })


Iteration: 0
AUC-ROC: Fold 0:0.9871638792482926


Iteration: 1
AUC-ROC: Fold 1:0.9904466834677931


Iteration: 2
AUC-ROC: Fold 2:0.9893480182615061


Iteration: 3
AUC-ROC: Fold 3:0.9931791906048912


Iteration: 4
AUC-ROC: Fold 4:0.9907309893289009


In [3]:
# Organize the results
df_results = pd.DataFrame(results)

# Display the results table
print("\n Cross-validation results (mean AUC-ROC for each value of k=100):\n")
print(df_results.to_string(index=False, float_format="%.4f"))


 Cross-validation results (mean AUC-ROC for each value of k=100):

 k_components  mean_auc_roc
          100        0.9902
